# 이상 자세 탐지 Autoencoder — Google Colab 학습
데이터 수집은 로컬(VS Code)에서 하고, 모델 학습은 Colab GPU를 사용하는 워크플로우입니다.

## 1. 환경 설정

In [ ]:
# 필요 패키지 설치
!pip install mediapipe torch torchvision matplotlib -q

In [ ]:
# Google Drive 마운트 (데이터 / 모델 저장용)
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트 폴더 경로 설정 (본인 경로로 수정)
PROJECT_DIR = '/content/drive/MyDrive/deeplearning_project/autoencoder_module'
DATA_PATH   = f'{PROJECT_DIR}/data/normal_poses.npy'
SAVE_DIR    = f'{PROJECT_DIR}/checkpoints'

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import os
os.chdir(PROJECT_DIR)
print('작업 디렉토리:', os.getcwd())

## 2. 데이터 확인

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load(DATA_PATH)
print(f'데이터 shape : {data.shape}')   # (N, 99) 이어야 함
print(f'평균값       : {data.mean():.4f}')
print(f'표준편차     : {data.std():.4f}')

# 첫 번째 샘플 시각화
plt.figure(figsize=(12, 3))
plt.plot(data[0])
plt.title('정상 자세 랜드마크 벡터 (샘플 1개)')
plt.xlabel('차원 (x0,y0,z0, x1,y1,z1, ..., x32,y32,z32)')
plt.ylabel('정규화된 좌표')
plt.tight_layout()
plt.show()

## 3. 모델 학습

In [ ]:
from train import train

threshold = train(
    data_path  = DATA_PATH,
    save_dir   = SAVE_DIR,
    epochs     = 150,      # Colab GPU 환경에서는 넉넉하게
    batch_size = 64,
    lr         = 1e-3,
    latent_dim = 16,
    threshold_percentile = 95.0,
)

print(f'\n최종 임계값: {threshold:.6f}')

## 4. 임계값 조정 (선택)
95th 백분위 기본값이 너무 민감하거나 둔감하면 아래에서 수동으로 재계산하세요.

In [ ]:
import torch
from model import PoseAutoencoder
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PoseAutoencoder(input_dim=99, latent_dim=16).to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/autoencoder_best.pth', map_location=device))
model.eval()

tensor = torch.tensor(data, dtype=torch.float32)
loader = DataLoader(TensorDataset(tensor), batch_size=256)
errors = []
with torch.no_grad():
    for (batch,) in loader:
        batch = batch.to(device)
        errors.extend(model.reconstruction_error(batch).cpu().numpy())
errors = np.array(errors)

# 백분위별 임계값 확인
for p in [90, 92, 95, 97, 99]:
    print(f'  {p}th 백분위: {np.percentile(errors, p):.6f}')

# 원하는 백분위로 재설정
NEW_PERCENTILE = 95  # ← 여기 수정
new_threshold = float(np.percentile(errors, NEW_PERCENTILE))
np.save(f'{SAVE_DIR}/threshold.npy', np.array([new_threshold]))
print(f'\n새 임계값 저장: {new_threshold:.6f}')